In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests

def run_ttest_for_age(age):
    outer_file = f"MLG_{age}_outer_label3_density.csv"
    ring_file = f"{age}_MLG_plaque_ring_MLGsubtype_density_per_mm2.csv"
    
    outer_df = pd.read_csv(outer_file)
    outer_df["density_per_um2"] = pd.to_numeric(outer_df["density_per_um2"], errors="coerce")
    outer_density = dict(zip(outer_df["states_nn_alg1_label3"], outer_df["density_per_um2"]))
    
    ring_df = pd.read_csv(
        ring_file, 
        header=None, 
        names=["id","ring","subtype","count","area_um2","density_per_mm2"]
    )
    ring_df["density_per_mm2"] = pd.to_numeric(ring_df["density_per_mm2"], errors="coerce")
    
    rings = sorted(ring_df['ring'].unique())
    labels3 = sorted(ring_df['subtype'].unique())
    
    results = []
    for ring in rings:
        for label3 in labels3:
            sub_df = ring_df[(ring_df['ring'] == ring) & (ring_df['subtype'] == label3)]
            densities = pd.to_numeric(sub_df['density_per_mm2'], errors='coerce').astype(float).values
            if len(densities) > 0:
                ref_density_raw = outer_density.get(label3, np.nan)
                if pd.isna(ref_density_raw):
                    continue
                ref_density_per_mm2 = float(ref_density_raw) * 1e6
                if np.isnan(ref_density_per_mm2):
                    continue
                t_stat, pval_two_sided = ttest_1samp(densities, ref_density_per_mm2, nan_policy='omit')
                if t_stat > 0:
                    pval_onesided = pval_two_sided / 2
                else:
                    pval_onesided = 1 - (pval_two_sided / 2)
                results.append({
                    "ring": ring,
                    "label3": label3,
                    "n": len(densities),
                    "mean_density_per_mm2": np.nanmean(densities),
                    "ref_density_per_mm2": ref_density_per_mm2,
                    "t_stat": t_stat,
                    "pval_oneside_gt": pval_onesided
                })
    result_df = pd.DataFrame(results)
    if not result_df.empty and "pval_oneside_gt" in result_df:
        adjusted = multipletests(result_df["pval_oneside_gt"], method='fdr_bh')
        result_df["pval_adj"] = adjusted[1]
    else:
        result_df["pval_adj"] = np.nan

    def pval_to_stars(p):
        if pd.isna(p):
            return ""
        elif p <= 0.0001:
            return "****"
        elif p <= 0.001:
            return "***"
        elif p <= 0.01:
            return "**"
        elif p <= 0.05:
            return "*"
        else:
            return ""
    result_df["pval_stars"] = result_df["pval_adj"].apply(pval_to_stars)
    return result_df

ages = ["4mAD", "8mAD", "14mAD"]

for age in ages:
    print(f"\n===== {age} =====")
    res = run_ttest_for_age(age)
    out_csv = f"{age}_ttest_onesample_vs_outer.csv"
    res[["ring", "label3", "n", "mean_density_per_mm2", "ref_density_per_mm2", "t_stat", "pval_oneside_gt", "pval_adj", "pval_stars"]].to_csv(out_csv, index=False)
    print(res[["ring", "label3", "n", "mean_density_per_mm2", "ref_density_per_mm2", "t_stat", "pval_oneside_gt", "pval_adj", "pval_stars"]].to_string(index=False))




===== 4mAD =====
         ring label3  n  mean_density_per_mm2  ref_density_per_mm2    t_stat  pval_oneside_gt     pval_adj pval_stars
 ring1_0_10um   MLG1 91            271.863864            19.806566  3.614424     2.478359e-04 1.239180e-03         **
 ring1_0_10um   MLG2 91            735.013837            49.334146  6.143630     1.077134e-08 1.615701e-07       ****
 ring1_0_10um   MLG3 91            309.520674             1.458152  3.628702     2.361902e-04 1.239180e-03         **
ring2_10_20um   MLG1 91             57.668048            19.806566  1.462075     7.360182e-02 1.326894e-01           
ring2_10_20um   MLG2 91            175.798416            49.334146  2.935131     2.115407e-03 6.346220e-03         **
ring2_10_20um   MLG3 91              9.987618             1.458152  0.854004     1.976851e-01 2.759372e-01           
ring3_20_30um   MLG1 91             31.266189            19.806566  0.837182     2.023539e-01 2.759372e-01           
ring3_20_30um   MLG2 91             83